Scrapping tunisie annonce offers

In [48]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_page(page_num):
    url = f"http://www.tunisie-annonce.com/AnnoncesImmobilier.asp?rech_cod_cat=1&rech_cod_rub=&rech_cod_typ=&rech_cod_sou_typ=&rech_cod_pay=TN&rech_cod_reg=&rech_cod_vil=&rech_cod_loc=&rech_prix_min=&rech_prix_max=&rech_surf_min=&rech_surf_max=&rech_age=&rech_photo=&rech_typ_cli=&rech_order_by=31&rech_page_num={page_num}"
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')

    # Find all rows with the class "Tableau1"
    rows = soup.find_all('tr', class_='Tableau1')

    # Extract data from each row
    data = []
    for row in rows:
        cells = row.find_all('td')
        if len(cells) > 10:  # Ensure there are enough cells in the row
            try:
                gouvernorat = cells[1].find('a').get('onmouseover').split('<br/>')[0].split(': ')[1]
                # Extract 'Nature' and 'Type' from the 4th and 5th <td> elements (onmouseover attributes)
                nature_info = cells[3].get('onmouseover', '')
                if nature_info:
                    try:
                        nature = nature_info.split('<br/>')[1].split(': ')[1].strip()
                    except IndexError:
                        nature = 'N/A'
                else:
                    nature = 'N/A'

                type_info = cells[5].get('onmouseover', '')
                if type_info:
                    try:
                        type_immobilier = type_info.split('<br/>')[2].split(': ')[1].strip()
                    except IndexError:
                        type_immobilier = 'N/A'
                else:
                    type_immobilier = 'N/A'

                # Extract 'post' URL
                a_tag = cells[7].find('a')
                url = 'N/A'

                # Check if the 'a' tag exists
                if a_tag:
                    # Extract the URL from the 'href' attribute
                    url = a_tag.get('href', 'N/A')

                # Extract 'Price' from the 9th <td> (text content)
                price_cell = cells[9]
                price = price_cell.get_text(strip=True)

                # Clean the price by removing unwanted characters (e.g., spaces or currency symbols)
                price = ''.join(filter(str.isdigit, price))  # Keep only digits


                # Adding the data to the list
                data.append({
                    'Gouvernorat': gouvernorat,
                    'Nature': nature,
                    'Type': type_immobilier,
                    'url': url,
                    'Price': price,
                })
            except (AttributeError, IndexError) as e:
                # Skip rows that do not match the expected structure or have missing attributes
                print(f"Error extracting data: {e}")
                continue

    return data

# Scrape data from the first page (1 to 1)
all_data = []
for page_num in range(1,246):  # Change this to a larger range if you need more pages
    all_data.extend(scrape_page(page_num))

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(all_data)
display(df)

'''# Save the DataFrame to a CSV file
output_file = "tunisie_annonce_data.csv"
df.to_csv(output_file, index=False, encoding='utf-8-sig')

# Print a success message and show the first few rows of the DataFrame
print(f"Data successfully saved to {output_file}")
print(df.head())'''


,Gouvernorat,Nature,Type,url,Price
0,Tunis,Location,Appart. 4 pièces');,DetailsAnnonceImmobilier.asp?cod_ann=3310087,3100
1,Tunis,Location,Appart. 4 pièces');,DetailsAnnonceImmobilier.asp?cod_ann=3310077,2700
2,Tunis,Location,Maisons');,DetailsAnnonceImmobilier.asp?cod_ann=3275712,6000
3,Tunis,Location,Maisons');,DetailsAnnonceImmobilier.asp?cod_ann=3154114,4500
4,Tunis,Location,Duplex');,DetailsAnnonceImmobilier.asp?cod_ann=3150447,3400
...,...,...,...,...,...
6120,Tunis,Location,Appart. 3 pièces');,DetailsAnnonceImmobilier.asp?cod_ann=3368712,550
6121,Tunis,Location,Appart. 2 pièces');,DetailsAnnonceImmobilier.asp?cod_ann=3372630,650
6122,Ariana,Terrain,Terrain nu');,DetailsAnnonceImmobilier.asp?cod_ann=3058964,4500000
6123,Ben arous,Vente,Appart. 3 pièces');,DetailsAnnonceImmobilier.asp?cod_ann=3370574,120000


'# Save the DataFrame to a CSV file\noutput_file = "tunisie_annonce_data.csv"\ndf.to_csv(output_file, index=False, encoding=\'utf-8-sig\')\n\n# Print a success message and show the first few rows of the DataFrame\nprint(f"Data successfully saved to {output_file}")\nprint(df.head())'

In [49]:
# Filter rows where 'Nature' column is equal to 'Vente'
df = df.loc[df['Nature'] == 'Vente']

In [50]:
# Filter rows with the desired format (singular and plural)
df = df[df['Type'].str.contains(r"^Appart\. \d+ pièces?'\);$", na=False)]

In [51]:
display(df)

,Gouvernorat,Nature,Type,url,Price
14,Ben arous,Vente,Appart. 3 pièces');,DetailsAnnonceImmobilier.asp?cod_ann=3339918,107000
17,Tunis,Vente,Appart. 4 pièces');,DetailsAnnonceImmobilier.asp?cod_ann=3374578,290000
31,Nabeul,Vente,Appart. 3 pièces');,DetailsAnnonceImmobilier.asp?cod_ann=3342662,690000
47,Tunis,Vente,Appart. 1 pièce');,DetailsAnnonceImmobilier.asp?cod_ann=3373523,245000
48,Ben arous,Vente,Appart. 4 pièces');,DetailsAnnonceImmobilier.asp?cod_ann=3311439,190000
...,...,...,...,...,...
6084,Tunis,Vente,Appart. 4 pièces');,DetailsAnnonceImmobilier.asp?cod_ann=3376431,110000
6086,Tunis,Vente,Appart. 3 pièces');,DetailsAnnonceImmobilier.asp?cod_ann=3376345,130000
6104,Ben arous,Vente,Appart. 3 pièces');,DetailsAnnonceImmobilier.asp?cod_ann=3365115,140000
6118,Ben arous,Vente,Appart. 3 pièces');,DetailsAnnonceImmobilier.asp?cod_ann=3364670,130000


In [52]:
# Regular Expression to Extract Number of Rooms
df['rooms'] = df['Type'].str.extract(r'(\d+)\s*pièce', expand=False).astype(float)

In [53]:
# Filter rows where Gouvernorat is one of the specified values
df = df[df['Gouvernorat'].isin(['Tunis', 'Ben Arous', 'Ariana', 'Manouba'])]

In [54]:
df= df.drop(columns=['Nature','Type'])

In [55]:
display(df)

,Gouvernorat,url,Price,rooms
17,Tunis,DetailsAnnonceImmobilier.asp?cod_ann=3374578,290000,4.0
47,Tunis,DetailsAnnonceImmobilier.asp?cod_ann=3373523,245000,1.0
73,Tunis,DetailsAnnonceImmobilier.asp?cod_ann=3371932,340000,3.0
92,Tunis,DetailsAnnonceImmobilier.asp?cod_ann=3324606,320000,3.0
96,Tunis,DetailsAnnonceImmobilier.asp?cod_ann=3372943,185000,4.0
...,...,...,...,...
6057,Tunis,DetailsAnnonceImmobilier.asp?cod_ann=3346904,1500000,4.0
6058,Tunis,DetailsAnnonceImmobilier.asp?cod_ann=3346903,1500000,4.0
6081,Tunis,DetailsAnnonceImmobilier.asp?cod_ann=3376434,265000,2.0
6084,Tunis,DetailsAnnonceImmobilier.asp?cod_ann=3376431,110000,4.0


Scrap every offer signal page

In [56]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_property_details(df):
    """
    This function takes a DataFrame containing URLs in the 'URL' column,
    scrapes the necessary details (localisation and description) from each URL,
    and returns a new DataFrame containing the scraped information.
    """
    # Initialize an empty list to store the results
    results = []

    # Iterate over each URL in the DataFrame
    for url in df['url']:
        try:
            # Make the request to the webpage
            response = requests.get("http://www.tunisie-annonce.com/"+url)
            response.raise_for_status()  # Ensure the request was successful

            # Parse the HTML content
            soup = BeautifulSoup(response.content, 'html.parser')

            # Extract the fields
            location = soup.find('td', class_='da_label_field', text='Localisation').find_next('td').get_text(strip=True)
            description = soup.find('td', class_='da_label_field', text='Texte').find_next('td').get_text(strip=True)

        except AttributeError:
            # In case of missing fields, assign 'N/A'
            location = "N/A"
            description = "N/A"
        except requests.exceptions.RequestException as e:
            # In case of any error with the request (e.g., 404, timeout)
            location = "N/A"
            description = "Error fetching data"

        # Append the results to the list
        results.append({
            "URL": url,
            "Localisation": location,
            "Description": description
        })

    # Convert the results list to a DataFrame
    df2 = pd.DataFrame(results)

    return df2

In [57]:
# Call the function to scrape data
df2 = scrape_property_details(df)

# Print the result DataFrame
display(df2)

<ipython-input-56-457e3aa84a6e>:25: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  location = soup.find('td', class_='da_label_field', text='Localisation').find_next('td').get_text(strip=True)
<ipython-input-56-457e3aa84a6e>:26: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  description = soup.find('td', class_='da_label_field', text='Texte').find_next('td').get_text(strip=True)


,URL,Localisation,Description
0,DetailsAnnonceImmobilier.asp?cod_ann=3374578,Tunisie>Tunis>La Marsa>El Aouina,ref.120a vendre un appartement s+3 d'une super...
1,DetailsAnnonceImmobilier.asp?cod_ann=3373523,Tunisie>Tunis>Ain Zaghouan>Ain Zaghouan,la croisette immobilière tunisie vous propose ...
2,DetailsAnnonceImmobilier.asp?cod_ann=3371932,Tunisie>Tunis>La Marsa>El Aouina,fb.7284contacter notre conseiller en immobilie...
3,DetailsAnnonceImmobilier.asp?cod_ann=3324606,Tunisie>Tunis>La Marsa>El Aouina,bonne affaire a ne pas rater a vendre appartem...
4,DetailsAnnonceImmobilier.asp?cod_ann=3372943,Tunisie>Tunis>El Ouerdia>El Ouerdia,immobilière al-amal vous prpose un appartement...
...,...,...,...
504,DetailsAnnonceImmobilier.asp?cod_ann=3346904,Tunisie>Tunis>La Marsa>Marsa Safsaf,arcane vous propose la location dun apparteme...
505,DetailsAnnonceImmobilier.asp?cod_ann=3346903,Tunisie>Tunis>La Marsa>Marsa Safsaf,arcane vous propose la location dun apparteme...
506,DetailsAnnonceImmobilier.asp?cod_ann=3376434,Tunisie>Tunis>Bab Bhar>Bab Bhar,à vendre appart s+2 73m2 3eme etage haut stand...
507,DetailsAnnonceImmobilier.asp?cod_ann=3376431,Tunisie>Tunis>Le Bardo>Ksar Said,"exclusivité immobilière jouini , ne manquez pa..."


In [58]:
# Merge df and df2 on the 'URL' column
merged_df = pd.merge(df, df2, left_on='url', right_on='URL', how='inner')
display(merged_df)

,Gouvernorat,url,Price,rooms,URL,Localisation,Description
0,Tunis,DetailsAnnonceImmobilier.asp?cod_ann=3374578,290000,4.0,DetailsAnnonceImmobilier.asp?cod_ann=3374578,Tunisie>Tunis>La Marsa>El Aouina,ref.120a vendre un appartement s+3 d'une super...
1,Tunis,DetailsAnnonceImmobilier.asp?cod_ann=3373523,245000,1.0,DetailsAnnonceImmobilier.asp?cod_ann=3373523,Tunisie>Tunis>Ain Zaghouan>Ain Zaghouan,la croisette immobilière tunisie vous propose ...
2,Tunis,DetailsAnnonceImmobilier.asp?cod_ann=3371932,340000,3.0,DetailsAnnonceImmobilier.asp?cod_ann=3371932,Tunisie>Tunis>La Marsa>El Aouina,fb.7284contacter notre conseiller en immobilie...
3,Tunis,DetailsAnnonceImmobilier.asp?cod_ann=3324606,320000,3.0,DetailsAnnonceImmobilier.asp?cod_ann=3324606,Tunisie>Tunis>La Marsa>El Aouina,bonne affaire a ne pas rater a vendre appartem...
4,Tunis,DetailsAnnonceImmobilier.asp?cod_ann=3372943,185000,4.0,DetailsAnnonceImmobilier.asp?cod_ann=3372943,Tunisie>Tunis>El Ouerdia>El Ouerdia,immobilière al-amal vous prpose un appartement...
...,...,...,...,...,...,...,...
504,Tunis,DetailsAnnonceImmobilier.asp?cod_ann=3346904,1500000,4.0,DetailsAnnonceImmobilier.asp?cod_ann=3346904,Tunisie>Tunis>La Marsa>Marsa Safsaf,arcane vous propose la location dun apparteme...
505,Tunis,DetailsAnnonceImmobilier.asp?cod_ann=3346903,1500000,4.0,DetailsAnnonceImmobilier.asp?cod_ann=3346903,Tunisie>Tunis>La Marsa>Marsa Safsaf,arcane vous propose la location dun apparteme...
506,Tunis,DetailsAnnonceImmobilier.asp?cod_ann=3376434,265000,2.0,DetailsAnnonceImmobilier.asp?cod_ann=3376434,Tunisie>Tunis>Bab Bhar>Bab Bhar,à vendre appart s+2 73m2 3eme etage haut stand...
507,Tunis,DetailsAnnonceImmobilier.asp?cod_ann=3376431,110000,4.0,DetailsAnnonceImmobilier.asp?cod_ann=3376431,Tunisie>Tunis>Le Bardo>Ksar Said,"exclusivité immobilière jouini , ne manquez pa..."


In [59]:
merged_df = merged_df.drop(columns=['URL','url'])

In [60]:
# Apply the operation to extract only the last part of the location
merged_df['Localisation'] = merged_df['Localisation'].apply(lambda x: x.split('>')[-1] if isinstance(x, str) else x)

In [61]:
# Drop rows where the 'Description' contains the word "villa"
merged_df = merged_df[~merged_df['Description'].str.contains('villa', case=False, na=False)]

In [69]:
# Drop rows where the 'Description' contains the word "terrain"
merged_df = merged_df[~merged_df['Description'].str.contains('terrain', case=False, na=False)]

In [70]:
display(merged_df)

,Gouvernorat,Price,rooms,Localisation,Description
0,Tunis,290000.0,4.0,El Aouina,ref.120a vendre un appartement s+3 d'une super...
1,Tunis,245000.0,1.0,Ain Zaghouan,la croisette immobilière tunisie vous propose ...
2,Tunis,340000.0,3.0,El Aouina,fb.7284contacter notre conseiller en immobilie...
3,Tunis,320000.0,3.0,El Aouina,bonne affaire a ne pas rater a vendre appartem...
4,Tunis,185000.0,4.0,El Ouerdia,immobilière al-amal vous prpose un appartement...
...,...,...,...,...,...
504,Tunis,1500000.0,4.0,Marsa Safsaf,arcane vous propose la location dun apparteme...
505,Tunis,1500000.0,4.0,Marsa Safsaf,arcane vous propose la location dun apparteme...
506,Tunis,265000.0,2.0,Bab Bhar,à vendre appart s+2 73m2 3eme etage haut stand...
507,Tunis,110000.0,4.0,Ksar Said,"exclusivité immobilière jouini , ne manquez pa..."


Checking Lower Prices

In [67]:
# Convert to float
merged_df['Price'] = merged_df['Price'].astype(float)

In [77]:
# Filter rows where the price is less than 90000
display(merged_df[merged_df['Price'] < 90000])

,Gouvernorat,Price,rooms,Localisation,Description
37,Tunis,65000.0,2.0,El Manar 1,l'agence krout immo mets à votre disposition u...
118,Tunis,60000.0,1.0,El Manar 1,"a vendre chez la rosa immobilière, un garage ..."
129,Tunis,85000.0,2.0,Bab Bhar,avenue hadi chakernotre agence eden immo met e...
227,Manouba,72000.0,3.0,Douar Hicher,a vendre appartement au 4ém étage emplacement ...
285,Manouba,50000.0,3.0,Denden,à el 3agba on vend un appartement propre de sa...
290,Tunis,75000.0,2.0,Bab El Khadra,on met a votre disposition la vente d'un appar...
318,Ariana,85000.0,4.0,Borj Louzir,à vendre : appartement au 3ème étage à borj lo...
372,Tunis,65000.0,2.0,Cite Ettahrir Sup.,l'agence krout immo mets à votre disposition u...
483,Manouba,85000.0,2.0,Oued Ellil,appartement s+2rue de chausséepetit jardinprem...


In [73]:
# Correct the price for two specific rows according to their description
merged_df.loc[243, 'Price'] = 174 * 3500
merged_df.loc[238, 'Price'] = 3500 * 124.5

In [76]:
# Multiply the price by 10 for rows where the price is under 50,000
merged_df.loc[merged_df['Price'] < 50000, 'Price'] *= 10

Checking Higher prices

In [78]:
# Filter rows where the price is higher than 500000
display(merged_df[merged_df['Price'] > 500000])

,Gouvernorat,Price,rooms,Localisation,Description
14,Tunis,550000.0,3.0,El Aouina,l'aouina a vendre appartement au rez de chauss...
19,Tunis,700000.0,4.0,Ain Zaghouan,sky immo propose à la vente un appartement en ...
22,Ariana,549000.0,4.0,La Soukra,la maison de limmobilier vous propose à la ve...
32,Tunis,638000.0,4.0,El Menzah 9,appartement s+3 à vendrehelma immo vous propos...
38,Tunis,585000.0,3.0,Ain Zaghouan,magnifique s3 de 139m² à aïn zaghouan nord. rd...
...,...,...,...,...,...
474,Tunis,520000.0,2.0,Jardins de Carthage,fb.6078contacter notre conseillère en immobili...
478,Tunis,620000.0,4.0,Jardins de Carthage,"un appartement s+3 de 198,47 m² sis dans une ..."
491,Tunis,650000.0,4.0,Ain Zaghouan,agence immobilière le carré 9 vous propose à l...
504,Tunis,1500000.0,4.0,Marsa Safsaf,arcane vous propose la location dun apparteme...


In [79]:
# Correct the price for two specific rows according to their description
merged_df.loc[44, 'Price'] /= 10

In [80]:
# Export the DataFrame to a CSV file
merged_df.to_csv('tunisie_annonce.csv', index=False)